# Door Condition Monitoring
## EDA and Temporal Segmentation

In [1]:
import pandas as pd

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

In [2]:
DATA_DIR = Path("../PS3/02_Datasets/Door")

TRAIN_PATH = DATA_DIR / "Train.csv"
SEGMENTS_PATH = DATA_DIR / "Train_Segments_Answer.csv"
TEST_PATH = DATA_DIR / "Test.csv"

print("Working directory:")
print(Path.cwd())

print("\nDataset paths:")
print(TRAIN_PATH.resolve())
print(SEGMENTS_PATH.resolve())
print(TEST_PATH.resolve())

print("\nFile checks:")
print("Train:", TRAIN_PATH.exists())
print("Segments:", SEGMENTS_PATH.exists())
print("Test:", TEST_PATH.exists())

Working directory:
/Users/yeo/Documents/Door/NOTEBOOKS

Dataset paths:
/Users/yeo/Documents/Door/Train.csv
/Users/yeo/Documents/Door/Train_Segments_Answer.csv
/Users/yeo/Documents/Door/Test.csv

File checks:
Train: True
Segments: True
Test: True


In [3]:
# Load the continuous training stream and its ground-truth segment labels.

train = pd.read_csv(TRAIN_PATH)
truth = pd.read_csv(SEGMENTS_PATH)

print("Train shape:", train.shape)
print("Ground-truth shape:", truth.shape)

Train shape: (18036, 17)
Ground-truth shape: (110, 6)


In [5]:
# inspect first few rows to understand raw structure for both

display(train.head())
display(truth.head())

,Datetime,Motor current(mA),Motor Voltage(10mV),Motor electrodynamic force,Door opening time(.1s),Door closing time(.1s),Close command,Open command,DCSR,DCSL,DLSR,DLSL,Door Opened,Door Locked,Door is opening,Door is closing,Door leaf position
0,2023-7-5-0-0-0-0,115,400,53,24,35,1,0,0,0,0,0,0,0,0,1,700
1,2023-7-5-0-0-0-20,132,600,93,24,35,1,0,0,0,0,0,0,0,0,1,700
2,2023-7-5-0-0-0-40,176,700,108,24,35,1,0,0,0,0,0,0,0,0,1,700
3,2023-7-5-0-0-0-60,317,1000,114,24,35,1,0,0,0,0,0,0,0,0,1,699
4,2023-7-5-0-0-0-80,503,1500,150,24,35,1,0,0,0,0,0,0,0,0,1,699


,segment_id,start_time,end_time,operation,status,n_rows
0,train_seg_001,2023-7-5-0-0-0-0,2023-7-5-0-0-3-700,Close,Normal,186
1,train_seg_002,2023-7-5-0-0-23-999,2023-7-5-0-0-26-839,Open,Normal,143
2,train_seg_003,2023-7-5-0-0-51-266,2023-7-5-0-0-53-986,Open,Abnormal resistance,137
3,train_seg_004,2023-7-5-0-1-33-989,2023-7-5-0-1-37-709,Close,Abnormal resistance,187
4,train_seg_005,2023-7-5-0-2-24-608,2023-7-5-0-2-28-308,Close,Abnormal resistance,186


In [7]:
# check missing values
missing = pd.DataFrame({"column": train.columns, "missing_count": train.isna().sum().values, "missing_percent": train.isna().mean().values * 100})

display(missing.sort_values("missing_count", ascending=False))

,column,missing_count,missing_percent
0,Datetime,0,0.0
9,DCSL,0,0.0
15,Door is closing,0,0.0
14,Door is opening,0,0.0
13,Door Locked,0,0.0
12,Door Opened,0,0.0
11,DLSL,0,0.0
10,DLSR,0,0.0
8,DCSR,0,0.0
1,Motor current(mA),0,0.0


In [8]:
# Count the number of operations by opening/closing type.

print("Operation distribution:")
display(truth["operation"].value_counts())

Operation distribution:


operation
Close    55
Open     55
Name: count, dtype: int64

In [9]:
# Count the Normal and Abnormal resistance operations.

print("Status distribution:")
display(truth["status"].value_counts())

Status distribution:


status
Normal                 80
Abnormal resistance    30
Name: count, dtype: int64

In [10]:
# Examine the joint distribution of operation type and fault status.

display(pd.crosstab(truth["operation"], truth["status"], margins=True))

status,Abnormal resistance,Normal,All
operation,,,
Close,15,40,55
Open,15,40,55
All,30,80,110


In [11]:
# Parse the dataset's Year-Month-Day-Hour-Minute-Second-Millisecond format.


def parse_door_datetime(value):
    parts = str(value).split("-")

    if len(parts) != 7:
        return pd.NaT

    try:
        year, month, day, hour, minute, second, ms = map(int, parts)

        return pd.Timestamp(year=year, month=month, day=day, hour=hour, minute=minute, second=second, microsecond=ms * 1000)

    except (ValueError, TypeError):
        return pd.NaT

In [12]:
# Convert both raw and ground-truth timestamps into Pandas timestamps.

train["timestamp"] = train["Datetime"].apply(parse_door_datetime)
truth["start_timestamp"] = truth["start_time"].apply(parse_door_datetime)
truth["end_timestamp"] = truth["end_time"].apply(parse_door_datetime)

print("Invalid Train timestamps:", train["timestamp"].isna().sum())
print("Invalid start timestamps:", truth["start_timestamp"].isna().sum())
print("Invalid end timestamps:", truth["end_timestamp"].isna().sum())

Invalid Train timestamps: 0
Invalid start timestamps: 0
Invalid end timestamps: 0


In [13]:
# Inspect the temporal coverage of the training stream.

print("First timestamp:", train["timestamp"].min())
print("Last timestamp: ", train["timestamp"].max())

print("\nTotal recording duration:", train["timestamp"].max() - train["timestamp"].min())

First timestamp: 2023-07-05 00:00:00
Last timestamp:  2023-07-05 01:10:17.112000

Total recording duration: 0 days 01:10:17.112000


In [14]:
# Calculate the actual duration of each ground-truth door operation.

truth["duration_seconds"] = (truth["end_timestamp"] - truth["start_timestamp"]).dt.total_seconds()

display(truth[["segment_id", "operation", "status", "duration_seconds", "n_rows"]].head(10))

,segment_id,operation,status,duration_seconds,n_rows
0,train_seg_001,Close,Normal,3.70,186
1,train_seg_002,Open,Normal,2.84,143
2,train_seg_003,Open,Abnormal resistance,2.72,137
3,train_seg_004,Close,Abnormal resistance,3.72,187
4,train_seg_005,Close,Abnormal resistance,3.70,186
5,train_seg_006,Open,Normal,2.86,144
6,train_seg_007,Open,Normal,2.78,140
7,train_seg_008,Open,Normal,2.88,145
8,train_seg_009,Close,Abnormal resistance,3.72,187
9,train_seg_010,Open,Normal,2.84,143


In [16]:
# Compare operation duration between Open and Close cycles.

display(truth.groupby("operation")["duration_seconds"].describe())

,count,mean,std,min,25%,50%,75%,max
operation,,,,,,,,
Close,55.0,3.698182,0.042735,3.54,3.68,3.70,3.72,3.78
Open,55.0,2.820364,0.041898,2.72,2.80,2.82,2.84,2.92


In [17]:
# Compare operation duration between Normal and Abnormal cycles.

display(truth.groupby("status")["duration_seconds"].describe())

,count,mean,std,min,25%,50%,75%,max
status,,,,,,,,
Abnormal resistance,30.0,3.270667,0.459467,2.72,2.825,3.31,3.72,3.78
Normal,80.0,3.255000,0.439453,2.76,2.820,3.21,3.70,3.78


In [18]:
# Check whether duration differs when operation and status are considered together.

display(truth.groupby(["operation", "status"])["duration_seconds"].describe())

count      mean       std   min   25%   50%   75%   max
operation status                                                                      
Close     Abnormal resistance   15.0  3.720000  0.021381  3.70  3.70  3.72  3.72  3.78
          Normal                40.0  3.690000  0.045965  3.54  3.68  3.70  3.70  3.78
Open      Abnormal resistance   15.0  2.821333  0.064793  2.72  2.78  2.82  2.86  2.92
          Normal                40.0  2.820000  0.030382  2.76  2.80  2.82  2.84  2.88

In [20]:
# Calculate the time elapsed between every pair of consecutive sensor readings.

train["dt_seconds"] = train["timestamp"].diff().dt.total_seconds()
display(train["dt_seconds"].describe())

# Show the most common time intervals between consecutive readings.

sampling_counts = train["dt_seconds"].round(6).value_counts().head(20)
display(sampling_counts)

count    18035.000000
mean         0.233829
std          2.947606
min          0.020000
25%          0.020000
50%          0.020000
75%          0.020000
max         58.823000
Name: dt_seconds, dtype: float64

dt_seconds
0.020     17926
20.299        1
24.427        1
40.003        1
46.899        1
30.769        1
54.786        1
38.297        1
24.313        1
37.273        1
52.435        1
29.802        1
16.241        1
21.728        1
15.918        1
22.995        1
23.981        1
42.030        1
36.987        1
43.520        1
Name: count, dtype: int64

In [21]:
# Inspect the largest timestamp gaps in the recording.

largest_gaps = train[["Datetime", "timestamp", "dt_seconds"]].sort_values("dt_seconds", ascending=False).head(20)
display(largest_gaps)

,Datetime,timestamp,dt_seconds
3516,2023-7-5-0-13-19-38,2023-07-05 00:13:19.038,58.823
17521,2023-7-5-1-8-24-815,2023-07-05 01:08:24.815,58.294
17663,2023-7-5-1-9-25-783,2023-07-05 01:09:25.783,58.148
9058,2023-7-5-0-36-41-950,2023-07-05 00:36:41.950,58.018
3982,2023-7-5-0-15-37-485,2023-07-05 00:15:37.485,56.946
6096,2023-7-5-0-25-10-422,2023-07-05 00:25:10.422,56.478
4594,2023-7-5-0-18-5-449,2023-07-05 00:18:05.449,55.169
15785,2023-7-5-1-2-22-708,2023-07-05 01:02:22.708,55.054
983,2023-7-5-0-3-56-723,2023-07-05 00:03:56.723,54.786
14621,2023-7-5-0-58-9-390,2023-07-05 00:58:09.390,54.424


In [22]:
# Compare the smallest large gaps against the normal sampling interval.

large_gaps = train.loc[train["dt_seconds"] > 0.020, "dt_seconds"]

print("Number of gaps > 20 ms:", len(large_gaps))
print("\nLarge-gap statistics:")
display(large_gaps.describe())

print("\nSmallest gap greater than 20 ms:", large_gaps.min())

Number of gaps > 20 ms: 109

Large-gap statistics:


count    109.000000
mean      35.399927
std       13.967978
min       10.215000
25%       24.313000
50%       36.987000
75%       46.899000
max       58.823000
Name: dt_seconds, dtype: float64


Smallest gap greater than 20 ms: 10.215


## Temporal Segmentation

In [24]:
# 1.0 being a conservative threshold to identify gaps between recorded door operations.

GAP_THRESHOLD_SECONDS = 1.0

gap_indices = train.index[train["dt_seconds"] > GAP_THRESHOLD_SECONDS].tolist()

print("Gap threshold:", GAP_THRESHOLD_SECONDS, "seconds")
print("Large gaps detected:", len(gap_indices))
print("Expected internal boundaries:", len(truth) - 1)

Gap threshold: 1.0 seconds
Large gaps detected: 109
Expected internal boundaries: 109


In [25]:
# above recognised 109 large gaps, which complements the fact that 110 true segments imply 109 internal boundaries

# Inspect the locations and timestamps of the detected gaps.

gap_details = train.loc[gap_indices, ["Datetime", "timestamp", "dt_seconds"]].copy()
display(gap_details.head(20))

,Datetime,timestamp,dt_seconds
186,2023-7-5-0-0-23-999,2023-07-05 00:00:23.999,20.299
329,2023-7-5-0-0-51-266,2023-07-05 00:00:51.266,24.427
466,2023-7-5-0-1-33-989,2023-07-05 00:01:33.989,40.003
653,2023-7-5-0-2-24-608,2023-07-05 00:02:24.608,46.899
839,2023-7-5-0-2-59-77,2023-07-05 00:02:59.077,30.769
983,2023-7-5-0-3-56-723,2023-07-05 00:03:56.723,54.786
1123,2023-7-5-0-4-37-800,2023-07-05 00:04:37.800,38.297
1268,2023-7-5-0-5-4-993,2023-07-05 00:05:04.993,24.313
1455,2023-7-5-0-5-45-986,2023-07-05 00:05:45.986,37.273
1598,2023-7-5-0-6-41-261,2023-07-05 00:06:41.261,52.435


In [26]:
# Convert the detected timestamp gaps into start and end row indices for each segment.

starts = [0] + gap_indices

ends = [index - 1 for index in gap_indices] + [len(train) - 1]

print("Number of starts:", len(starts))
print("Number of ends:", len(ends))

Number of starts: 110
Number of ends: 110


In [27]:
# Build a segment table using the row boundaries detected from timestamp gaps.

detected = pd.DataFrame({"start_idx": starts, "end_idx": ends})
detected["n_rows"] = detected["end_idx"] - detected["start_idx"] + 1
detected["start_timestamp"] = detected["start_idx"].map(train["timestamp"])
detected["end_timestamp"] = detected["end_idx"].map(train["timestamp"])

display(detected.head(10))

,start_idx,end_idx,n_rows,start_timestamp,end_timestamp
0,0,185,186,2023-07-05 00:00:00.000,2023-07-05 00:00:03.700
1,186,328,143,2023-07-05 00:00:23.999,2023-07-05 00:00:26.839
2,329,465,137,2023-07-05 00:00:51.266,2023-07-05 00:00:53.986
3,466,652,187,2023-07-05 00:01:33.989,2023-07-05 00:01:37.709
4,653,838,186,2023-07-05 00:02:24.608,2023-07-05 00:02:28.308
5,839,982,144,2023-07-05 00:02:59.077,2023-07-05 00:03:01.937
6,983,1122,140,2023-07-05 00:03:56.723,2023-07-05 00:03:59.503
7,1123,1267,145,2023-07-05 00:04:37.800,2023-07-05 00:04:40.680
8,1268,1454,187,2023-07-05 00:05:04.993,2023-07-05 00:05:08.713
9,1455,1597,143,2023-07-05 00:05:45.986,2023-07-05 00:05:48.826


In [28]:
# Confirm that the number of detected segments matches the number of labelled operations.

print("Detected segments:", len(detected))
print("Ground-truth segments:", len(truth))

if len(detected) == len(truth):
    print("Segment count matches ground truth.")
else:
    print("Segment count does not match ground truth.")

Detected segments: 110
Ground-truth segments: 110
Segment count matches ground truth.


In [29]:
# Compare each detected segment with the corresponding ground-truth segment.

if len(detected) == len(truth):
    segmentation_comparison = pd.DataFrame(
        {
            "true_start": truth["start_timestamp"],
            "pred_start": detected["start_timestamp"],
            "true_end": truth["end_timestamp"],
            "pred_end": detected["end_timestamp"],
            "true_n_rows": truth["n_rows"],
            "pred_n_rows": detected["n_rows"],
        }
    )

    segmentation_comparison["start_error_ms"] = (segmentation_comparison["pred_start"] - segmentation_comparison["true_start"]).dt.total_seconds() * 1000
    segmentation_comparison["end_error_ms"] = (segmentation_comparison["pred_end"] - segmentation_comparison["true_end"]).dt.total_seconds() * 1000
    segmentation_comparison["row_error"] = segmentation_comparison["pred_n_rows"] - segmentation_comparison["true_n_rows"]

    display(segmentation_comparison.head(20))

else:
    print("Cannot compare segment-by-segment because the counts differ.")

,true_start,pred_start,true_end,pred_end,true_n_rows,pred_n_rows,start_error_ms,end_error_ms,row_error
0,2023-07-05 00:00:00.000,2023-07-05 00:00:00.000,2023-07-05 00:00:03.700,2023-07-05 00:00:03.700,186,186,0.0,0.0,0
1,2023-07-05 00:00:23.999,2023-07-05 00:00:23.999,2023-07-05 00:00:26.839,2023-07-05 00:00:26.839,143,143,0.0,0.0,0
2,2023-07-05 00:00:51.266,2023-07-05 00:00:51.266,2023-07-05 00:00:53.986,2023-07-05 00:00:53.986,137,137,0.0,0.0,0
3,2023-07-05 00:01:33.989,2023-07-05 00:01:33.989,2023-07-05 00:01:37.709,2023-07-05 00:01:37.709,187,187,0.0,0.0,0
4,2023-07-05 00:02:24.608,2023-07-05 00:02:24.608,2023-07-05 00:02:28.308,2023-07-05 00:02:28.308,186,186,0.0,0.0,0
5,2023-07-05 00:02:59.077,2023-07-05 00:02:59.077,2023-07-05 00:03:01.937,2023-07-05 00:03:01.937,144,144,0.0,0.0,0
6,2023-07-05 00:03:56.723,2023-07-05 00:03:56.723,2023-07-05 00:03:59.503,2023-07-05 00:03:59.503,140,140,0.0,0.0,0
7,2023-07-05 00:04:37.800,2023-07-05 00:04:37.800,2023-07-05 00:04:40.680,2023-07-05 00:04:40.680,145,145,0.0,0.0,0
8,2023-07-05 00:05:04.993,2023-07-05 00:05:04.993,2023-07-05 00:05:08.713,2023-07-05 00:05:08.713,187,187,0.0,0.0,0
9,2023-07-05 00:05:45.986,2023-07-05 00:05:45.986,2023-07-05 00:05:48.826,2023-07-05 00:05:48.826,143,143,0.0,0.0,0


In [30]:
# Summarise how accurately the timestamp-gap method reproduces the labelled boundaries.

if len(detected) == len(truth):
    print("Start-time error (ms):")
    display(segmentation_comparison["start_error_ms"].describe())

    print("\nEnd-time error (ms):")
    display(segmentation_comparison["end_error_ms"].describe())

    print("\nRow-count error:")
    display(segmentation_comparison["row_error"].describe())

Start-time error (ms):


count    110.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: start_error_ms, dtype: float64


End-time error (ms):


count    110.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: end_error_ms, dtype: float64


Row-count error:


count    110.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: row_error, dtype: float64

In [31]:
# Count how many segments have exactly matching starts, ends, and row counts.

if len(detected) == len(truth):
    exact_start = (segmentation_comparison["start_error_ms"] == 0).sum()
    exact_end = (segmentation_comparison["end_error_ms"] == 0).sum()
    exact_rows = (segmentation_comparison["row_error"] == 0).sum()

    total = len(segmentation_comparison)

    print(f"Exact start matches: {exact_start}/{total}")
    print(f"Exact end matches:   {exact_end}/{total}")
    print(f"Exact row matches:   {exact_rows}/{total}")

Exact start matches: 110/110
Exact end matches:   110/110
Exact row matches:   110/110


# Temporal IoU Check

In [32]:
# Calculate temporal Intersection-over-Union between a true and detected segment.


def temporal_iou(true_start, true_end, pred_start, pred_end):
    intersection_start = max(true_start, pred_start)

    intersection_end = min(true_end, pred_end)

    intersection = max(0, (intersection_end - intersection_start).total_seconds())

    true_duration = (true_end - true_start).total_seconds()

    pred_duration = (pred_end - pred_start).total_seconds()

    union = true_duration + pred_duration - intersection

    if union <= 0:
        return 0.0

    return intersection / union

In [33]:
# Calculate the IoU for every detected segment.

if len(detected) == len(truth):
    segmentation_comparison["iou"] = segmentation_comparison.apply(
        lambda row: temporal_iou(row["true_start"], row["true_end"], row["pred_start"], row["pred_end"]), axis=1
    )

    display(segmentation_comparison.head(20))

,true_start,pred_start,true_end,pred_end,true_n_rows,pred_n_rows,start_error_ms,end_error_ms,row_error,iou
0,2023-07-05 00:00:00.000,2023-07-05 00:00:00.000,2023-07-05 00:00:03.700,2023-07-05 00:00:03.700,186,186,0.0,0.0,0,1.0
1,2023-07-05 00:00:23.999,2023-07-05 00:00:23.999,2023-07-05 00:00:26.839,2023-07-05 00:00:26.839,143,143,0.0,0.0,0,1.0
2,2023-07-05 00:00:51.266,2023-07-05 00:00:51.266,2023-07-05 00:00:53.986,2023-07-05 00:00:53.986,137,137,0.0,0.0,0,1.0
3,2023-07-05 00:01:33.989,2023-07-05 00:01:33.989,2023-07-05 00:01:37.709,2023-07-05 00:01:37.709,187,187,0.0,0.0,0,1.0
4,2023-07-05 00:02:24.608,2023-07-05 00:02:24.608,2023-07-05 00:02:28.308,2023-07-05 00:02:28.308,186,186,0.0,0.0,0,1.0
5,2023-07-05 00:02:59.077,2023-07-05 00:02:59.077,2023-07-05 00:03:01.937,2023-07-05 00:03:01.937,144,144,0.0,0.0,0,1.0
6,2023-07-05 00:03:56.723,2023-07-05 00:03:56.723,2023-07-05 00:03:59.503,2023-07-05 00:03:59.503,140,140,0.0,0.0,0,1.0
7,2023-07-05 00:04:37.800,2023-07-05 00:04:37.800,2023-07-05 00:04:40.680,2023-07-05 00:04:40.680,145,145,0.0,0.0,0,1.0
8,2023-07-05 00:05:04.993,2023-07-05 00:05:04.993,2023-07-05 00:05:08.713,2023-07-05 00:05:08.713,187,187,0.0,0.0,0,1.0
9,2023-07-05 00:05:45.986,2023-07-05 00:05:45.986,2023-07-05 00:05:48.826,2023-07-05 00:05:48.826,143,143,0.0,0.0,0,1.0


In [34]:
# Summarise the temporal overlap achieved by the segmentation method.

if "iou" in segmentation_comparison:
    print("IoU statistics:")
    display(segmentation_comparison["iou"].describe())

    print("\nMean IoU:", segmentation_comparison["iou"].mean())

    print("Minimum IoU:", segmentation_comparison["iou"].min())

    print("Perfect IoU matches:", (segmentation_comparison["iou"] == 1.0).sum(), "/", len(segmentation_comparison))

IoU statistics:


count    110.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: iou, dtype: float64


Mean IoU: 1.0
Minimum IoU: 1.0
Perfect IoU matches: 110 / 110


## Create the Segmented Training Dataset

In [35]:
# Attach the official labels to our detected segment boundaries.

if len(detected) != len(truth):
    raise ValueError("Detected segment count does not match ground truth.")

segments = detected.copy()

segments["segment_id"] = truth["segment_id"].values
segments["operation"] = truth["operation"].values
segments["status"] = truth["status"].values

segments["duration_seconds"] = (segments["end_timestamp"] - segments["start_timestamp"]).dt.total_seconds()

display(segments.head(10))

,start_idx,end_idx,n_rows,start_timestamp,end_timestamp,segment_id,operation,status,duration_seconds
0,0,185,186,2023-07-05 00:00:00.000,2023-07-05 00:00:03.700,train_seg_001,Close,Normal,3.70
1,186,328,143,2023-07-05 00:00:23.999,2023-07-05 00:00:26.839,train_seg_002,Open,Normal,2.84
2,329,465,137,2023-07-05 00:00:51.266,2023-07-05 00:00:53.986,train_seg_003,Open,Abnormal resistance,2.72
3,466,652,187,2023-07-05 00:01:33.989,2023-07-05 00:01:37.709,train_seg_004,Close,Abnormal resistance,3.72
4,653,838,186,2023-07-05 00:02:24.608,2023-07-05 00:02:28.308,train_seg_005,Close,Abnormal resistance,3.70
5,839,982,144,2023-07-05 00:02:59.077,2023-07-05 00:03:01.937,train_seg_006,Open,Normal,2.86
6,983,1122,140,2023-07-05 00:03:56.723,2023-07-05 00:03:59.503,train_seg_007,Open,Normal,2.78
7,1123,1267,145,2023-07-05 00:04:37.800,2023-07-05 00:04:40.680,train_seg_008,Open,Normal,2.88
8,1268,1454,187,2023-07-05 00:05:04.993,2023-07-05 00:05:08.713,train_seg_009,Close,Abnormal resistance,3.72
9,1455,1597,143,2023-07-05 00:05:45.986,2023-07-05 00:05:48.826,train_seg_010,Open,Normal,2.84


In [36]:
# Verify that the segmented training table preserves the original class distribution.

display(pd.crosstab(segments["operation"], segments["status"], margins=True))

status,Abnormal resistance,Normal,All
operation,,,
Close,15,40,55
Open,15,40,55
All,30,80,110


In [37]:
# Save the segmented training metadata for reuse in later notebooks.

OUTPUT_PATH = Path("segmented_training_cycles.csv")

segments.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH.resolve())
print("Shape:", segments.shape)

Saved: /Users/yeo/Documents/Door/NOTEBOOKS/segmented_training_cycles.csv
Shape: (110, 9)
